# Parameter Recovery v4: Motor Noise Estimation + Partial Fitting

**Approach:**
1. Simulate responses with known ground-truth parameters
2. Estimate σ_motor from lateral error using ITD+ILD model (von Mises ML)
3. Fit remaining parameters (σ_spectral, σ_prior) with σ_motor fixed
4. Evaluate recovery: compare recovered vs ground truth

**Outputs:** `results/parameter_recovery_results.csv` and `results/parameter_recovery_simulations.pkl`
(Sec. 3.3, Fig. S3 via `figures/fig_parameter_recovery.py`). If both files exist they are loaded and
the (slow) recovery is skipped.

*Saved outputs* come from the run used for the paper (development version of `bayesian_listener`
preceding 0.2.0, original SONICOM KEMAR GRAS large-ears file). The last section regenerates responses
from the recovered parameters; re-running it with 0.2.0 gives statistically equivalent, not identical,
numbers.

In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfar as pf
import pickle
import time
import warnings
from pathlib import Path
from scipy.special import i0, i1
from scipy.optimize import minimize_scalar
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')

from bayesian_listener import BayesianListener
from bayesian_listener.fitting import fit_listener_partial, sigma_to_kappa, kappa_to_sigma

import paths

## Configuration

In [49]:
# KEMAR HRTF SOFA file (SONICOM dataset), shipped with the repository
sofa_path = str(paths.KEMAR_SOFA)
CACHE_DIR = paths.CACHE_DIR

# Parameter combinations (28 sets, uncorrelated, coherent sampling)
# - sigma_spectral: 2-25 (log-spaced)
# - sigma_prior: 5-90 (log-spaced)
# - sigma_motor: 5-16 (linear)
# Max |r_spearman| < 0.1 between all parameter pairs
PARAM_COMBINATIONS = [
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.0, 'sigma_prior': 22.4, 'sigma_motor': 7.9},  # rmsP=14.2 rmsL=7.8 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.2, 'sigma_prior': 6.9, 'sigma_motor': 14.0},  # rmsP=36.1 rmsL=14.1 qe=6.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.3, 'sigma_prior': 42.5, 'sigma_motor': 5.0},  # rmsP=8.9 rmsL=5.5 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.5, 'sigma_prior': 58.7, 'sigma_motor': 15.2},  # rmsP=18.8 rmsL=15.3 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.7, 'sigma_prior': 18.1, 'sigma_motor': 13.1},  # rmsP=23.0 rmsL=13.4 qe=1.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.9, 'sigma_prior': 13.1, 'sigma_motor': 16.0},  # rmsP=33.4 rmsL=17.2 qe=2.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.1, 'sigma_prior': 20.1, 'sigma_motor': 5.4},  # rmsP=24.1 rmsL=6.7 qe=1.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.4, 'sigma_prior': 65.3, 'sigma_motor': 9.5},  # rmsP=20.1 rmsL=10.5 qe=4.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.6, 'sigma_prior': 30.9, 'sigma_motor': 7.0},  # rmsP=26.1 rmsL=8.1 qe=3.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.9, 'sigma_prior': 34.3, 'sigma_motor': 5.8},  # rmsP=23.5 rmsL=6.6 qe=6.4
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 4.2, 'sigma_prior': 38.2, 'sigma_motor': 14.4},  # rmsP=26.7 rmsL=14.4 qe=2.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 4.5, 'sigma_prior': 27.7, 'sigma_motor': 11.9},  # rmsP=27.9 rmsL=12.8 qe=8.8
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 4.9, 'sigma_prior': 14.6, 'sigma_motor': 9.1},  # rmsP=34.2 rmsL=9.4 qe=9.9
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 5.3, 'sigma_prior': 8.5, 'sigma_motor': 13.6},  # rmsP=41.5 rmsL=14.1 qe=13.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 5.7, 'sigma_prior': 52.7, 'sigma_motor': 11.1},  # rmsP=28.0 rmsL=12.7 qe=8.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 6.1, 'sigma_prior': 47.3, 'sigma_motor': 12.7},  # rmsP=33.1 rmsL=13.7 qe=13.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 6.6, 'sigma_prior': 6.2, 'sigma_motor': 6.6},  # rmsP=40.1 rmsL=10.3 qe=18.6
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 7.1, 'sigma_prior': 7.7, 'sigma_motor': 6.2},  # rmsP=38.6 rmsL=10.5 qe=28.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 7.7, 'sigma_prior': 24.9, 'sigma_motor': 11.5},  # rmsP=31.3 rmsL=11.8 qe=13.8
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 8.3, 'sigma_prior': 5.0, 'sigma_motor': 15.6},  # rmsP=34.4 rmsL=15.6 qe=16.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 8.9, 'sigma_prior': 5.6, 'sigma_motor': 12.3},  # rmsP=34.7 rmsL=15.1 qe=26.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 9.6, 'sigma_prior': 16.2, 'sigma_motor': 9.9},  # rmsP=35.0 rmsL=12.2 qe=18.2
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 10.3, 'sigma_prior': 10.6, 'sigma_motor': 8.7},  # rmsP=34.7 rmsL=11.4 qe=23.5
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 11.1, 'sigma_prior': 80.9, 'sigma_motor': 14.8},  # rmsP=37.4 rmsL=15.8 qe=24.1
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 12.0, 'sigma_prior': 90.0, 'sigma_motor': 10.7},  # rmsP=35.4 rmsL=12.7 qe=18.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 12.9, 'sigma_prior': 11.8, 'sigma_motor': 8.3},  # rmsP=39.3 rmsL=12.8 qe=29.1
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 13.9, 'sigma_prior': 72.7, 'sigma_motor': 7.4},  # rmsP=35.6 rmsL=10.9 qe=13.9
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 15.0, 'sigma_prior': 9.5, 'sigma_motor': 10.3},  # rmsP=43.4 rmsL=12.5 qe=24.1
]

TRIALS_PER_TARGET = 3
MC_SAMPLES_FIT = 200
MC_SAMPLES_MOTOR = 200
N_GRID_POINTS = 1
BASE_SEED = 42
RUN_PARALLEL = True
N_JOBS = -1

PARAMS_TO_FIT = ['sigma_spectral', 'sigma_prior']

print(f'Testing {len(PARAM_COMBINATIONS)} parameter combinations')
print(f'Fitting: {PARAMS_TO_FIT}')
print(f'Motor noise: estimated from ITD+ILD')

Testing 28 parameter combinations
Fitting: ['sigma_spectral', 'sigma_prior']
Motor noise: estimated from ITD+ILD


## Target grid

In [50]:
TARGET_GRID = np.array([
    [315,  30], [315, -30], [180, -30], [ 90,  30],
    [120,   0], [ 90,   0], [225,  30], [330,   0],
    [270, -30], [  0, -30], [180,  30], [135,  30],
    [270,  30], [300,   0], [ 30,   0], [240,   0],
    [ 60,   0], [225, -30], [135, -30], [  0,  90],
    [210,   0], [270,  60], [  0,   0], [  0,  30],
    [ 90,  60], [180,  60], [180,   0], [270,   0],
    [ 45,  30], [ 45, -30], [  0,  60], [150,   0],
    [ 90, -30],
])

_tg_arr = np.column_stack([
    np.deg2rad(TARGET_GRID[:, 0]),
    np.deg2rad(TARGET_GRID[:, 1]),
    np.ones(len(TARGET_GRID)),
])
target_coords = pf.Coordinates.from_spherical_elevation(_tg_arr[:, 0], _tg_arr[:, 1], _tg_arr[:, 2])

print(f'{len(TARGET_GRID)} targets, {len(TARGET_GRID) * TRIALS_PER_TARGET} trials per simulation')

33 targets, 99 trials per simulation


## Helper functions

In [51]:
def wrap_to_pi(rad):
    return (rad + np.pi) % (2 * np.pi) - np.pi

def von_mises_loglik_mc(kappa, resp_lat, est_lat_mc):
    """Negative log-likelihood for von Mises with MC estimates."""
    log_C = -np.log(2 * np.pi * i0(kappa))
    lat_diff = resp_lat[:, None] - est_lat_mc
    log_pdfs = log_C + kappa * np.cos(lat_diff)
    max_log_pdfs = np.max(log_pdfs, axis=1, keepdims=True)
    log_mean_probs = max_log_pdfs[:, 0] + np.log(np.mean(np.exp(log_pdfs - max_log_pdfs), axis=1))
    return -np.sum(log_mean_probs)

def fit_kappa_ml(resp_lat, est_lat_mc):
    result = minimize_scalar(von_mises_loglik_mc, bounds=(0.1, 500),
                            args=(resp_lat, est_lat_mc), method='bounded')
    return result.x

def simulate_dataset(model, target_coords, gt_params, trials_per_target, seed):
    """Generate simulated responses."""
    target_indices = model.coords.find_nearest(target_coords)[0][0]
    full_target = model.target
    model.target = full_target[target_indices]   # simulate the 33 experimental directions only

    kappa_motor = sigma_to_kappa(gt_params['sigma_motor'])
    model.parameters = {k: v for k, v in gt_params.items() if k != 'sigma_motor'} | {'kappa_motor': kappa_motor}
    posterior = model.infer(repetitions=trials_per_target, seed=seed)
    estimations = model.estimate(posterior, kappa_motor=kappa_motor, seed=seed).cartesian
    model.target = full_target

    rows = []
    target_sph = target_coords.spherical_elevation

    for t_idx in range(len(target_indices)):
        resp_cart = estimations[t_idx, :, :]
        resp_cart = resp_cart / np.linalg.norm(resp_cart, axis=1, keepdims=True)
        resp_coords = pf.Coordinates.from_cartesian(resp_cart[:, 0], resp_cart[:, 1], resp_cart[:, 2])
        resp_sph = resp_coords.spherical_elevation

        for i in range(trials_per_target):
            rows.append({
                'azi_target': np.rad2deg(target_sph[t_idx, 0]),
                'ele_target': np.rad2deg(target_sph[t_idx, 1]),
                'azi_response': np.rad2deg(resp_sph[i, 0]),
                'ele_response': np.rad2deg(resp_sph[i, 1]),
            })

    return pd.DataFrame(rows)

def estimate_motor_noise(model, obs_df, target_coords, n_mc, seed):
    """Estimate motor noise using ITD+ILD model."""
    np.random.seed(seed)

    sigma_itd = 0.569
    sigma_ild = 1.0

    # Template ITD+ILD
    template_itd = model.template.itd.flatten()
    template_ild = model.template.ild.flatten()
    template_hp = model.template.coords.spherical_side
    template_lat = template_hp[:, 0]

    # Target ITD+ILD
    target_idx = model.coords.find_nearest(target_coords)[0][0]
    target_itd = model.target.itd[target_idx].flatten()
    target_ild = model.target.ild[target_idx].flatten()

    # Response lateral angles
    _resp_arr = np.column_stack([np.deg2rad(obs_df['azi_response'].values),
                                   np.deg2rad(obs_df['ele_response'].values),
                                   np.ones(len(obs_df))])
    resp_coords = pf.Coordinates.from_spherical_elevation(_resp_arr[:, 0], _resp_arr[:, 1], _resp_arr[:, 2])
    resp_lat = resp_coords.spherical_side[:, 0]

    # Trial -> target mapping
    _targ_arr = np.column_stack([np.deg2rad(obs_df['azi_target'].values),
                                   np.deg2rad(obs_df['ele_target'].values),
                                   np.ones(len(obs_df))])
    targ_coords = pf.Coordinates.from_spherical_elevation(_targ_arr[:, 0], _targ_arr[:, 1], _targ_arr[:, 2])
    trial_target_indices = target_coords.find_nearest(targ_coords)[0][0]

    # MC estimates for each trial
    n_trials = len(obs_df)
    est_lat_mc = np.zeros((n_trials, n_mc))

    for i, t_idx in enumerate(trial_target_indices):
        noisy_itd = target_itd[t_idx] + np.random.normal(0, sigma_itd, n_mc)
        noisy_ild = target_ild[t_idx] + np.random.normal(0, sigma_ild, n_mc)

        itd_diff = (template_itd[None, :] - noisy_itd[:, None]) / sigma_itd
        ild_diff = (template_ild[None, :] - noisy_ild[:, None]) / sigma_ild
        loglik = -0.5 * (itd_diff**2 + ild_diff**2)

        best_indices = np.argmax(loglik, axis=1)
        est_lat_mc[i, :] = template_lat[best_indices]

    # Filter to ±80° lateral
    mask = np.abs(resp_lat) <= np.deg2rad(80)
    resp_lat_filt = resp_lat[mask]
    est_lat_mc_filt = est_lat_mc[mask, :]

    # Fit kappa
    kappa_fit = fit_kappa_ml(resp_lat_filt, est_lat_mc_filt)
    sigma_motor = kappa_to_sigma(kappa_fit)

    return sigma_motor, kappa_fit

print('Helper functions defined')

Helper functions defined


## Run recovery

In [52]:
def run_single_recovery(combo_idx, gt_params, sofa_path, target_coords,
                        trials_per_target, mc_samples_fit, mc_samples_motor,
                        n_grid_points, params_to_fit, base_seed):
    """Run full recovery pipeline for one parameter combination."""
    seed = base_seed + combo_idx * 100

    # Load model
    model = BayesianListener(sofa_path)
    model.compute_template(interpolation='SHMAX', cache_dir=CACHE_DIR)

    # 1. Simulate data
    obs_df = simulate_dataset(model, target_coords, gt_params, trials_per_target, seed)

    # 2. Estimate motor noise from ITD+ILD
    sigma_motor_est, kappa_est = estimate_motor_noise(
        model, obs_df, target_coords, mc_samples_motor, seed + 1000)

    # 3. Fit remaining parameters with estimated motor noise fixed
    fixed_params = {
        'kappa_motor': kappa_est,              # motor noise fixed at the stage-1 estimate
        'sigma_ild': gt_params['sigma_ild'],  # Keep ILD fixed at true value
    }

    fitted = fit_listener_partial(
        sofa_path=sofa_path,
        obs_tbl=obs_df,
        targets_coords=target_coords,
        interpolation_method='SHMAX',
        params_to_fit=params_to_fit,
        fixed_params=fixed_params,
        subject_id=None,
        num_repetitions=mc_samples_fit,
        num_grid_points=n_grid_points,
        verbose=False
    )

    # Collect results
    result = {'combo_idx': combo_idx, 'seed': seed, 'obs_df': obs_df}

    # Motor noise
    result['sigma_motor_true'] = gt_params['sigma_motor']
    result['sigma_motor_recovered'] = sigma_motor_est
    result['kappa_motor'] = kappa_est

    # Fitted parameters
    for param in params_to_fit:
        result[f'{param}_true'] = gt_params[param]
        result[f'{param}_recovered'] = fitted[param]

    result['nll'] = fitted['nll']

    return result

# ---------- Paths ----------
results_csv_path = paths.RESULTS_DIR / 'parameter_recovery_results.csv'
simdata_pkl_path = paths.RESULTS_DIR / 'parameter_recovery_simulations.pkl'
results_csv_path.parent.mkdir(parents=True, exist_ok=True)

# ---------- Load cached or run ----------
if results_csv_path.exists() and simdata_pkl_path.exists():
    print(f'Loading cached results from {results_csv_path.relative_to(paths.ROOT)}')
    results_df = pd.read_csv(results_csv_path)
    with open(simdata_pkl_path, 'rb') as f:  # only unpickle files from trusted sources
        simulated_data = pickle.load(f)
    print(f'Loaded {len(results_df)} results and {len(simulated_data)} simulated datasets')
else:
    # Run recovery
    t_start = time.time()

    if RUN_PARALLEL:
        print(f'Running {len(PARAM_COMBINATIONS)} combinations in PARALLEL...')
        results = Parallel(n_jobs=N_JOBS, verbose=10)(
            delayed(run_single_recovery)(
                i, gt, sofa_path, target_coords,
                TRIALS_PER_TARGET, MC_SAMPLES_FIT, MC_SAMPLES_MOTOR,
                N_GRID_POINTS, PARAMS_TO_FIT, BASE_SEED
            )
            for i, gt in enumerate(PARAM_COMBINATIONS)
        )
    else:
        print(f'Running {len(PARAM_COMBINATIONS)} combinations SEQUENTIALLY...')
        results = []
        for i, gt in enumerate(PARAM_COMBINATIONS):
            print(f'\nCombo {i+1}/{len(PARAM_COMBINATIONS)}: σ_spec={gt["sigma_spectral"]}, σ_prior={gt["sigma_prior"]}, σ_motor={gt["sigma_motor"]}')
            r = run_single_recovery(
                i, gt, sofa_path, target_coords,
                TRIALS_PER_TARGET, MC_SAMPLES_FIT, MC_SAMPLES_MOTOR,
                N_GRID_POINTS, PARAMS_TO_FIT, BASE_SEED
            )
            print(f'  Motor: {r["sigma_motor_true"]:.1f} -> {r["sigma_motor_recovered"]:.1f}')
            print(f'  Spectral: {r["sigma_spectral_true"]:.1f} -> {r["sigma_spectral_recovered"]:.1f}')
            print(f'  Prior: {r["sigma_prior_true"]:.1f} -> {r["sigma_prior_recovered"]:.1f}')
            results.append(r)

    t_total = time.time() - t_start

    # Store obs_df separately, create results_df without it
    simulated_data = {r['combo_idx']: r['obs_df'] for r in results}
    results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'obs_df'} for r in results])

    # Save results
    results_df.to_csv(results_csv_path, index=False)
    with open(simdata_pkl_path, 'wb') as f:
        pickle.dump(simulated_data, f)
    print(f'\nCompleted in {t_total:.1f}s ({t_total/len(PARAM_COMBINATIONS):.1f}s each)')
    print(f'Saved: {results_csv_path.relative_to(paths.ROOT)}, {simdata_pkl_path.relative_to(paths.ROOT)}')

Loading cached results from results/parameter_recovery_results.csv
Loaded 28 results and 28 simulated datasets


## Evaluation

In [53]:
from scipy import stats

params_to_eval = ['sigma_motor'] + PARAMS_TO_FIT

summary = []
for param in params_to_eval:
    true_vals = results_df[f'{param}_true'].values
    rec_vals = results_df[f'{param}_recovered'].values

    r, p = stats.pearsonr(true_vals, rec_vals)
    r2 = r ** 2
    bias = rec_vals - true_vals

    summary.append({
        'Parameter': param,
        'R²': r2,
        'r': r,
        'p-value': p,
        'Mean Bias': np.mean(bias),
        'Std Bias': np.std(bias),
        'Mean Rel. Bias (%)': 100 * np.mean(bias / true_vals),
    })

summary_df = pd.DataFrame(summary)
print('=== Parameter Recovery Summary ===')
print(summary_df.to_string(index=False, float_format='%.3f'))

print('\n=== Recovery Quality ===')
for _, row in summary_df.iterrows():
    r2 = row['R²']
    quality = 'GOOD' if r2 >= 0.80 else ('MODERATE' if r2 >= 0.50 else 'POOR')
    print(f"  {row['Parameter']:<18}: R²={r2:.3f} ({quality}), bias={row['Mean Bias']:+.2f}")

=== Parameter Recovery Summary ===
     Parameter    R²     r  p-value  Mean Bias  Std Bias  Mean Rel. Bias (%)
   sigma_motor 0.937 0.968    0.000     -0.135     0.872              -1.700
sigma_spectral 0.728 0.853    0.000      2.980     5.225              39.575
   sigma_prior 0.712 0.844    0.000     19.939    29.495              65.451

=== Recovery Quality ===
  sigma_motor       : R²=0.937 (GOOD), bias=-0.14
  sigma_spectral    : R²=0.728 (MODERATE), bias=+2.98
  sigma_prior       : R²=0.712 (MODERATE), bias=+19.94


In [55]:
# Detailed results
print('=== Detailed Results ===')
cols = ['combo_idx']
for p in params_to_eval:
    cols.extend([f'{p}_true', f'{p}_recovered'])
cols.append('nll')

print(results_df[cols].to_string(index=False, float_format='%.2f'))


=== Detailed Results ===
 combo_idx  sigma_motor_true  sigma_motor_recovered  sigma_spectral_true  sigma_spectral_recovered  sigma_prior_true  sigma_prior_recovered     nll
         0              7.90                   7.17                 2.00                      3.22             22.40                  37.03  -86.08
         1             14.00                  14.49                 2.20                      4.04              6.90                  13.78   53.13
         2              5.00                   4.69                 2.30                      2.58             42.50                  29.49 -151.07
         3             15.20                  14.62                 2.50                      3.79             58.70                 157.27   46.11
         4             13.10                  11.37                 2.70                      3.77             18.10                  92.90   15.88
         5             16.00                  17.03                 2.90               

## Localization Metrics Comparison

Compare metrics between:
- **Simulated data**: The responses used for fitting (generated with GT parameters)
- **Recovered model**: New responses generated with recovered parameters

In [57]:
import bayesian_listener.metrics as metrics_module
from scipy import stats

def compute_metrics_from_df(obs_df, target_coords):
    """Compute localization metrics from obs_df (azi/ele response/target in degrees)."""
    targets_hp = target_coords.spherical_side

    # Response coordinates
    _resp_arr = np.column_stack([
        np.deg2rad(obs_df['azi_response'].values),
        np.deg2rad(obs_df['ele_response'].values),
        np.ones(len(obs_df))
    ])
    resp_coords = pf.Coordinates.from_spherical_elevation(_resp_arr[:, 0], _resp_arr[:, 1], _resp_arr[:, 2])
    resp_hp = resp_coords.spherical_side

    # Target indices
    _targ_arr = np.column_stack([
        np.deg2rad(obs_df['azi_target'].values),
        np.deg2rad(obs_df['ele_target'].values),
        np.ones(len(obs_df))
    ])
    targ_coords = pf.Coordinates.from_spherical_elevation(_targ_arr[:, 0], _targ_arr[:, 1], _targ_arr[:, 2])
    target_indices = target_coords.find_nearest(targ_coords)[0][0]
    targ_hp = targets_hp[target_indices, :]

    rmsL, _ = metrics_module.METRIC_FUNCTIONS['rmsL'](targ_hp, resp_hp)
    rmsPlocal, _ = metrics_module.METRIC_FUNCTIONS['rmsPmedianlocal'](targ_hp, resp_hp)
    querr, _ = metrics_module.METRIC_FUNCTIONS['querrMiddlebrooks'](targ_hp, resp_hp)

    return {
        'rmsL_deg': np.rad2deg(rmsL),
        'rmsPlocal_deg': np.rad2deg(rmsPlocal),
        'querr': querr
    }

def generate_responses_df(model, target_coords, params, n_reps, seed):
    """Generate synthetic responses and return as DataFrame."""
    target_indices = model.coords.find_nearest(target_coords)[0][0]
    full_target = model.target
    model.target = full_target[target_indices]
    target_sph = target_coords.spherical_elevation

    model.parameters = params.copy()
    posterior = model.infer(repetitions=n_reps, seed=seed)
    estimations = model.estimate(posterior, kappa_motor=params['kappa_motor'], seed=seed).cartesian
    model.target = full_target

    rows = []
    for t_idx in range(len(target_indices)):
        resp_cart = estimations[t_idx, :, :]
        resp_cart = resp_cart / np.linalg.norm(resp_cart, axis=1, keepdims=True)
        resp_coords = pf.Coordinates.from_cartesian(resp_cart[:, 0], resp_cart[:, 1], resp_cart[:, 2])
        resp_sph = resp_coords.spherical_elevation

        for i in range(n_reps):
            rows.append({
                'azi_target': np.rad2deg(target_sph[t_idx, 0]),
                'ele_target': np.rad2deg(target_sph[t_idx, 1]),
                'azi_response': np.rad2deg(resp_sph[i, 0]),
                'ele_response': np.rad2deg(resp_sph[i, 1]),
            })

    return pd.DataFrame(rows)

print('Metrics functions defined')

Metrics functions defined


In [59]:
# Compute metrics: simulated data (used for fitting) vs recovered model predictions
model = BayesianListener(sofa_path)
model.compute_template(interpolation='SHMAX', cache_dir=CACHE_DIR)

metrics_rows = []

for i, gt_params in enumerate(PARAM_COMBINATIONS):
    # Get the simulated data that was used for fitting
    obs_df_gt = simulated_data[i]

    # Recovered parameters
    row = results_df[results_df['combo_idx'] == i].iloc[0]
    rec_params = {
        'sigma_itd': gt_params['sigma_itd'],
        'sigma_ild': gt_params['sigma_ild'],
        'sigma_spectral': row['sigma_spectral_recovered'],
        'sigma_prior': row['sigma_prior_recovered'],
        'kappa_motor': sigma_to_kappa(row['sigma_motor_recovered']),
    }

    # Metrics from simulated data (GT params)
    m_gt = compute_metrics_from_df(obs_df_gt, target_coords)

    # Generate new data with recovered params and compute metrics
    seed_rec = BASE_SEED + i * 100 + 5000
    obs_df_rec = generate_responses_df(model, target_coords, rec_params, TRIALS_PER_TARGET, seed_rec)
    m_rec = compute_metrics_from_df(obs_df_rec, target_coords)

    metrics_rows.append({
        'combo_idx': i,
        'rmsL_gt': m_gt['rmsL_deg'],
        'rmsL_rec': m_rec['rmsL_deg'],
        'rmsPlocal_gt': m_gt['rmsPlocal_deg'],
        'rmsPlocal_rec': m_rec['rmsPlocal_deg'],
        'querr_gt': m_gt['querr'],
        'querr_rec': m_rec['querr'],
    })

    print(f"Combo {i}: rmsL={m_gt['rmsL_deg']:.1f}→{m_rec['rmsL_deg']:.1f}, "
          f"rmsPlocal={m_gt['rmsPlocal_deg']:.1f}→{m_rec['rmsPlocal_deg']:.1f}, "
          f"querr={m_gt['querr']:.1f}→{m_rec['querr']:.1f}")

metrics_df = pd.DataFrame(metrics_rows)

✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
Combo 0: rmsL=7.8→7.3, rmsPlocal=19.9→13.9, querr=0.0→2.2
Combo 1: rmsL=15.2→13.9, rmsPlocal=29.4→37.7, querr=10.0→6.7
Combo 2: rmsL=5.2→5.8, rmsPlocal=10.3→19.2, querr=0.0→3.2
Combo 3: rmsL=14.7→16.0, rmsPlocal=20.5→25.1, querr=2.0→2.0
Combo 4: rmsL=12.3→11.9, rmsPlocal=23.7→16.4, querr=0.0→2.2
Combo 5: rmsL=17.4→17.2, rmsPlocal=26.3→28.9, querr=2.0→2.0
Combo 6: rmsL=5.7→6.1, rmsPlocal=21.0→25.2, querr=0.0→2.8
Combo 7: rmsL=9.1→8.2, rmsPlocal=14.3→22.4, querr=2.3→2.3
Combo 8: rmsL=8.4→8.8, rmsPlocal=27.6→15.9, querr=0.0→10.0
Combo 9: rmsL=5.8→5.9, rmsPlocal=29.2→20.9, querr=5.3→5.6
Combo 10: rmsL=12.8→12.9, rmsPlocal=28.0→24.4, querr=0.0→14.0
Combo 11: rmsL=12.6→11.2, rmsPlocal=28.9→37.4, querr=9.5→20.0
Combo 12: rmsL=9.6→9.2, rmsPlocal=34.1→28.9, querr=18.4→9.1
Combo 13: rmsL=13.7→15.8, rmsPlocal=44.2→32.3, querr=18.6→19.0
Combo 14: rmsL=10.9→10

In [60]:
# Metrics comparison: correlation and bias
print('=== Localization Metrics: Ground-Truth vs Recovered ===\n')

metric_names = ['rmsL', 'rmsPlocal', 'querr']
metric_labels = ['Lateral RMS (°)', 'Polar RMS local (°)', 'Quadrant Error (%)']

metrics_summary = []
for m, label in zip(metric_names, metric_labels):
    gt_vals = metrics_df[f'{m}_gt'].values
    rec_vals = metrics_df[f'{m}_rec'].values

    r, p = stats.pearsonr(gt_vals, rec_vals)
    r2 = r ** 2
    diff = rec_vals - gt_vals

    metrics_summary.append({
        'Metric': label,
        'r': r,
        'p-val': p,
        'R²': r2,
        'Mean GT': np.mean(gt_vals),
        'Mean Rec': np.mean(rec_vals),
        'Mean Diff': np.mean(diff),
        'Std Diff': np.std(diff),
    })

metrics_summary_df = pd.DataFrame(metrics_summary)
#print(metrics_summary_df.to_string(index=False, float_format='%.2f'))

print(metrics_summary_df.to_latex(index=False, float_format='%.2f'))

=== Localization Metrics: Ground-Truth vs Recovered ===

\begin{tabular}{lrrrrrrr}
\toprule
Metric & r & p-val & R² & Mean GT & Mean Rec & Mean Diff & Std Diff \\
\midrule
Lateral RMS (°) & 0.97 & 0.00 & 0.95 & 11.09 & 11.05 & -0.04 & 0.79 \\
Polar RMS local (°) & 0.72 & 0.00 & 0.52 & 32.06 & 31.71 & -0.34 & 6.92 \\
Quadrant Error (%) & 0.80 & 0.00 & 0.64 & 13.21 & 15.30 & 2.08 & 6.75 \\
\bottomrule
\end{tabular}

